In [1]:
import pandas as pd
import numpy as np
import json
import joblib
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm


In [2]:
# --- 1. Define Paths to Artifacts ---
# These paths point to the input directories from your attached Kaggle Datasets.
MODEL_PATH = '/kaggle/input/modelfined/my_finetuned_model'
MLB_PATH = '/kaggle/input/modelfined/label_binarizer.joblib'
CONFUSION_DICT_PATH = '/kaggle/input/confusion/confusion_dictionary.json'
TEST_DATA_PATH = '/kaggle/input/map-charting-student-math-misunderstandings/test.csv'
mod = '/kaggle/input/modelfined/my_finetuned_model'

In [3]:
tokenizer = AutoTokenizer.from_pretrained(mod)
model = AutoModelForSequenceClassification.from_pretrained(mod)

2025-09-14 16:38:05.742172: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757867885.950355      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757867886.010452      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
mlb = joblib.load(MLB_PATH)
labels = mlb.classes_

In [5]:
with open(CONFUSION_DICT_PATH, 'r') as f:
    confusion_dict = json.load(f)


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval() # Set model to evaluation mode

print("Artifacts loaded successfully.")

Artifacts loaded successfully.


In [7]:
def predict_from_model(text, row_id_debug=None):
    """
    Predict top 3 labels directly from the fine-tuned model
    without using confusion dictionary.
    """
    # Tokenize
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Model forward pass
    with torch.no_grad():
        logits = model(**inputs).logits[0]
    
    # Probabilities
    probabilities = torch.sigmoid(logits).cpu().numpy()
    ranked_indices = np.argsort(probabilities)[::-1]
    
    # Top 3 predictions
    top_labels = [labels[i] for i in ranked_indices[:3]]
    
    if row_id_debug is not None and row_id_debug < 5:
        print(f"Debug Row {row_id_debug}: {top_labels}")
    
    return top_labels

# --- 4. Load Test Data ---
print("Loading test data...")
test_df = pd.read_csv(TEST_DATA_PATH)


test_df['input_text'] = test_df.apply(
    lambda row: f"Question: {row.QuestionText} Answer: {row.MC_Answer} Explanation: {row.StudentExplanation}", 
    axis=1
)

# --- 5. Generate Predictions ---
print("Generating predictions...")
all_predictions = []
for i, text in enumerate(tqdm(test_df['input_text'].tolist(), desc="Predicting")):
    preds = predict_from_model(text, row_id_debug=i)
    all_predictions.append(preds)

# --- 6. Save Submission ---
print("Formatting and saving submission file...")
submission_df = pd.DataFrame({
    'row_id': test_df['row_id'],
    'Category:Misconception': [' '.join(pred_list).replace(':nan', ':NA') for pred_list in all_predictions]
})

submission_df.to_csv('submission.csv', index=False)

print("="*50)
print("submission.csv created successfully!")
print("="*50)
print(submission_df.head())

Loading test data...
Generating predictions...


Predicting:   0%|          | 0/3 [00:00<?, ?it/s]

Debug Row 0: ['True_Correct:nan', 'True_Neither:nan', 'True_Misconception:Firstterm']
Debug Row 1: ['False_Misconception:WNB', 'False_Neither:nan', 'False_Misconception:Incomplete']
Debug Row 2: ['True_Neither:nan', 'True_Correct:nan', 'True_Misconception:Shorter_is_bigger']
Formatting and saving submission file...
submission.csv created successfully!
   row_id                             Category:Misconception
0   36696  True_Correct:NA True_Neither:NA True_Misconcep...
1   36697  False_Misconception:WNB False_Neither:NA False...
2   36698  True_Neither:NA True_Correct:NA True_Misconcep...


In [8]:
for i, text in enumerate(tqdm(test_df['input_text'].tolist(), desc="Predicting")):
    preds = predict_from_model(text, row_id_debug=i)
    
    # DEBUG: print top 5 raw probabilities for first 3 rows
    if i < 3:
        with torch.no_grad():
            inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
            logits = model(**inputs).logits[0]
            probs = torch.sigmoid(logits).cpu().numpy()
            print(f"\nRow {i} probs snapshot:")
            for idx in np.argsort(probs)[::-1][:5]:
                print(f"  {labels[idx]} -> {probs[idx]:.4f}")
    
    all_predictions.append(preds)


Predicting:   0%|          | 0/3 [00:00<?, ?it/s]

Debug Row 0: ['True_Correct:nan', 'True_Neither:nan', 'True_Misconception:Firstterm']

Row 0 probs snapshot:
  True_Correct:nan -> 0.9778
  True_Neither:nan -> 0.2600
  True_Misconception:Firstterm -> 0.0018
  True_Misconception:Shorter_is_bigger -> 0.0017
  True_Misconception:Incomplete -> 0.0014
Debug Row 1: ['False_Misconception:WNB', 'False_Neither:nan', 'False_Misconception:Incomplete']

Row 1 probs snapshot:
  False_Misconception:WNB -> 0.9990
  False_Neither:nan -> 0.2681
  False_Misconception:Incomplete -> 0.0506
  False_Misconception:Adding_terms -> 0.0316
  True_Neither:nan -> 0.0300
Debug Row 2: ['True_Neither:nan', 'True_Correct:nan', 'True_Misconception:Shorter_is_bigger']

Row 2 probs snapshot:
  True_Neither:nan -> 0.9908
  True_Correct:nan -> 0.1140
  True_Misconception:Shorter_is_bigger -> 0.0413
  True_Misconception:Firstterm -> 0.0104
  True_Misconception:Tacking -> 0.0090


In [9]:
print("MLB Classes:", labels[:10])


MLB Classes: ['False_Correct:nan' 'False_Misconception:Adding_across'
 'False_Misconception:Adding_terms' 'False_Misconception:Additive'
 'False_Misconception:Base_rate' 'False_Misconception:Certainty'
 'False_Misconception:Definition'
 'False_Misconception:Denominator-only_change'
 'False_Misconception:Division' 'False_Misconception:Duplication']


In [10]:
submission_df

,row_id,Category:Misconception
0,36696,True_Correct:NA True_Neither:NA True_Misconcep...
1,36697,False_Misconception:WNB False_Neither:NA False...
2,36698,True_Neither:NA True_Correct:NA True_Misconcep...
